In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

RANDOM_STATE = 42

# Data input 

In [2]:
df = pd.read_csv(r"D:\CityU\5001\project\data\BRFSS_2023_ML_Feature_Dataset.csv")

X = df.drop(columns=["SCD", "Dataset_Split"])
y = df["SCD"].astype(int)

train_mask = df["Dataset_Split"] == "train"
X_train, X_test = X[train_mask].copy(), X[~train_mask].copy()
y_train, y_test = y[train_mask].copy(), y[~train_mask].copy()

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"SCD prevalence  train: {y_train.mean():.3f}  test: {y_test.mean():.3f}")

Train: (28603, 25), Test: (7151, 25)
SCD prevalence  train: 0.171  test: 0.171


# Preprocess

In [3]:
binary_vars = ["SEXVAR", "Depression_bin", "Stroke_bin", "HighBP_bin",
               "PhysicalActivity_bin", "ACEDEPRS_bin", "ACEDRINK_bin",
               "ACEDRUGS_bin", "ACEPRISN_bin", "ACEDIVRC_bin",
               "ACEPUNCH_bin", "ACEHURT1_bin", "ACESWEAR_bin",
               "ACETOUCH_bin", "ACETTHEM_bin", "ACEHVSEX_bin"]
numeric_vars = ["_AGE80"]
categorical_vars = ["_RACEGR3", "_STATE", "EDUCA", "INCOME3", "MARITAL",
                    "_BMI5CAT", "DIABETE4", "_SMOKER3"]

preprocessor = ColumnTransformer([
    ("bin", SimpleImputer(strategy="most_frequent"), binary_vars),
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_vars),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_vars),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Logistic regression

In [4]:
logit_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=2000,
                               random_state=RANDOM_STATE)),
])

logit_grid = {
    "clf__C": [0.01, 0.1, 1, 10],
    "clf__penalty": ["l2"],
    "clf__solver": ["lbfgs"],
}

logit_search = GridSearchCV(logit_pipe, logit_grid, scoring="f1",
                            cv=cv, n_jobs=-1)
logit_search.fit(X_train, y_train)

print("Best params:", logit_search.best_params_)
print(f"Best CV F1: {logit_search.best_score_:.4f}")

d:\Software\python\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Best params: {'clf__C': 0.1, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs'}
Best CV F1: 0.3984


# Random forest

In [5]:
rf_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(class_weight="balanced",
                                   random_state=RANDOM_STATE, n_jobs=-1)),
])

rf_grid = {
    "clf__n_estimators": [200, 400],
    "clf__max_depth": [None, 10, 20],
    "clf__min_samples_split": [2, 10],
}

rf_search = GridSearchCV(rf_pipe, rf_grid, scoring="f1",
                         cv=cv, n_jobs=-1)
rf_search.fit(X_train, y_train)

print("Best params:", rf_search.best_params_)
print(f"Best CV F1: {rf_search.best_score_:.4f}")

Best params: {'clf__max_depth': 10, 'clf__min_samples_split': 2, 'clf__n_estimators': 400}
Best CV F1: 0.4024


# Test set estimate

In [6]:
def evaluate(model, name):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    return {
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, pred), 4),
        "Precision": round(precision_score(y_test, pred), 4),
        "Recall": round(recall_score(y_test, pred), 4),
        "F1": round(f1_score(y_test, pred), 4),
        "ROC_AUC": round(roc_auc_score(y_test, prob), 4),
    }

metrics = pd.DataFrame([
    evaluate(logit_search.best_estimator_, "Logistic Regression"),
    evaluate(rf_search.best_estimator_, "Random Forest"),
])
metrics

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.6968,0.2976,0.5683,0.3907,0.7065
1,Random Forest,0.7269,0.3193,0.5274,0.3978,0.7117


# Output

In [7]:
import os

# 1) Test-set performance metrics (to Member C for the three-model comparison)
metrics.to_csv("B_model_performance.csv", index=False)

# 2) Best hyperparameters and CV scores (for the Methods section of the report)
best_params = pd.DataFrame([
    {"Model": "Logistic Regression",
     "Best_Params": str(logit_search.best_params_),
     "Best_CV_F1": round(logit_search.best_score_, 4)},
    {"Model": "Random Forest",
     "Best_Params": str(rf_search.best_params_),
     "Best_CV_F1": round(rf_search.best_score_, 4)},
])
best_params.to_csv("B_best_params.csv", index=False)

# 3) Train/test split assignments (redundant backup; C reads Dataset_Split directly)
pd.DataFrame({
    "row_index": df.index,
    "Dataset_Split": df["Dataset_Split"]
}).to_csv("B_split_assignments.csv", index=False)

# 4) Test-set predictions (to Member D for confusion matrices and ROC curves)
predictions = pd.DataFrame({
    "y_true": y_test.values,
    "logit_pred": logit_search.predict(X_test),
    "logit_prob": logit_search.predict_proba(X_test)[:, 1],
    "rf_pred": rf_search.predict(X_test),
    "rf_prob": rf_search.predict_proba(X_test)[:, 1],
})
predictions.to_csv("B_test_predictions.csv", index=False)

# 5) Fitted model objects (to Member D for SHAP and feature importance)
joblib.dump(logit_search.best_estimator_, "B_logistic_model.joblib")
joblib.dump(rf_search.best_estimator_, "B_random_forest_model.joblib")

print("Saved files:")
for f in sorted(os.listdir(".")):
    if f.startswith("B_"):
        print(" -", f)

Saved files:
 - B_best_params.csv
 - B_logistic_model.joblib
 - B_model_performance.csv
 - B_random_forest_model.joblib
 - B_split_assignments.csv
 - B_test_predictions.csv
